# Lesson 5 — Building a Guarded AI Data Analyst Agent

## Learning Objective

In this lesson, I will build a simple AI Data Analyst Agent with input and output guardrails.

The agent will:

- check whether the user's question is related to data analysis
- limit the number of invalid attempts
- use Python/Pandas to calculate the result
- send the verified result to Gemini
- check Gemini's response before showing it to the user

### Key Principle

Python calculates and verifies the data.

Gemini explains the verified result.

Guardrails control the input and output of the agent.


## Step 1 — Create the Practice Dataset

### Question

Which region has the highest sales?

### Purpose

We will create a small dataset containing regions and their sales values.

Python will later calculate which region has the highest sales.

In [1]:
import pandas as pd

data = {
    'Region': ['North', 'South', 'West', 'East'],
    'Sales': [10000, 20000, 15000, 9000]
}

df = pd.DataFrame(data)

df

,Region,Sales
0,North,10000
1,South,20000
2,West,15000
3,East,9000


### Interpretation

The dataset contains four regions with different sales values.

The South region currently has the highest sales value of 20,000, but we will use Python to calculate and verify this rather than manually selecting the answer.

### Insight

The agent should use Python for numerical calculations so that the LLM does not have to calculate the result itself.

## Step 2 — Calculate the Verified Result

### Question

Which region has the highest sales?

### Approach

Instead of asking the LLM to calculate the highest sales, we will use Pandas to find the row containing the maximum sales value.

This result will become the verified source that we later provide to Gemini.

In [2]:
highest_sales = df.loc[df['Sales'].idxmax()]

highest_sales

,1
Region,South
Sales,20000


### Interpretation

`idxmax()` identifies the position of the highest value in the Sales column.

`df.loc[]` then retrieves the complete row containing that value.

The verified result is:

- Region: South
- Sales: 20,000

### Insight

`highest_sales` contains the complete row, not just the sales number. Therefore, we can extract both the Region and Sales values from it.

## Step 3 — Create the Structured Verified Result

### Question

How can we pass only the required verified information to the LLM?

### Approach

We will create a dictionary containing:

- the region with the highest sales
- the corresponding sales value

This keeps the result simple and makes it clear that Python has already performed the calculation.

In [3]:
result = {
    'region': highest_sales['Region'],
    'sales': highest_sales['Sales']
}

result

{'region': 'South', 'sales': np.int64(20000)}

### Interpretation

The Python calculation has produced a structured, verified result:

- Region: South
- Sales: 20,000

This result will be provided to Gemini as the factual basis for its explanation.

### Insight

Python is responsible for calculating the answer, while the LLM will later be responsible only for explaining the verified result in natural language.

## Step 4 — Build the Input Guardrail

### Question

How can we prevent unrelated questions from entering the AI Data Analyst Agent?

### Approach

The input guardrail checks the user's question before the agent performs its analysis.

It looks for words related to data analysis, such as:

- sales
- profit
- revenue
- customer
- region
- product
- data
- analysis

If a relevant word is found, the request is accepted.

If no relevant word is found, the request is rejected.

### Insight

The input guardrail acts as a control point before the main agent workflow begins.

In [4]:
def input_guardrail(question):

    allowed_words = [
        'sales',
        'profit',
        'revenue',
        'customer',
        'region',
        'products',
        'product',
        'data',
        'analysis'
    ]

    question = question.lower()

    for word in allowed_words:
        if word in question:
            return True

    return False

### Test the Input Guardrail

We will test both a valid data-analysis question and an unrelated question.

Expected behavior:

- Data-analysis question → `True`
- Unrelated question → `False`

In [5]:
print(input_guardrail("What is the highest sales region?"))
print(input_guardrail("Tell me a joke"))

True
False


### Interpretation

The guardrail correctly accepts the data-analysis question and rejects the unrelated question.

### Insight

The input guardrail prevents the agent from processing requests that are outside its intended purpose.

## Step 5 — Limit Invalid Attempts

### Question

What should happen if the user repeatedly enters questions that fail the input guardrail?

### Approach

The agent will allow a maximum of 3 invalid attempts.

- `attempts` keeps track of failed attempts.
- `max_attempts` defines the maximum allowed attempts.
- `continue` skips the rest of the current loop and asks for another question.
- `break` stops the program when the maximum number of attempts is reached.

### Insight

The attempt limit prevents the agent from continuing indefinitely when users repeatedly provide invalid requests.

In [8]:
attempts = 0
max_attempts = 3

while attempts < max_attempts:

    question = input("Ask a data analysis question (type exit to exit): ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    if not input_guardrail(question):

        attempts += 1
        remaining = max_attempts - attempts

        if remaining > 0:
            print("Request rejected.")
            print(f"Remaining attempts: {remaining}")
            continue

        else:
            print("Maximum attempts reached. Program stopped.")
            break

    print("Request accepted.")
    break

Ask a data analysis question (type exit to exit): exit
Goodbye!


### Interpretation

The loop gives the user up to three attempts when the input guardrail rejects a question.

`continue` allows the user to try again.

`break` stops the loop when:

1. The user enters `exit`.
2. The maximum number of invalid attempts is reached.
3. A valid request is accepted in this practice test.

### Insight

Loop control is important in an AI agent because a failed request should not cause the program to run forever.

## Step 6 — Send the Verified Result to Gemini

### Question

How can the AI explain the result without performing the calculation itself?

### Approach

Python has already calculated the highest-sales region and stored the verified result.

We will send this structured result to Gemini and ask it to explain the finding in one simple sentence.

Gemini will not be asked to calculate the highest sales.

### Insight

The agent separates two responsibilities:

- Python → calculation and verification
- Gemini → natural-language explanation

In [9]:
from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get('GEMINI_API_KEY')
)

### Create the Prompt

The prompt will provide Gemini with the verified Python result.

The model will be instructed not to change or invent the values.

In [10]:
prompt = f"""
You are a data analyst.

The Python calculation produced this verified result:
{result}

Explain the finding in one simple sentence.
Do not change or invent the region or sales value.
"""

In [11]:
response = client.models.generate_content(
    model='gemini-flash-latest',
    contents=prompt
)

print(response.text)

The total sales for the South region reached 20,000.


### Validation Note

At this stage, the Gemini response is printed only to inspect the raw model output during testing.

In the final integrated workflow, the response is checked by the output guardrail before it is shown to the user.

### Interpretation

Gemini generated a natural-language explanation using the verified Python result.

The model was not responsible for calculating the highest sales region.

### Insight

This separation reduces the risk of the LLM inventing or incorrectly calculating numerical results.

The verified Python result remains the source of truth.

## Step 7 — Build the Output Guardrail

### Question

How can we prevent Gemini from returning an answer that does not match the verified Python result?

### Approach

The output guardrail will compare Gemini's response with the verified Python result.

It will check two things:

1. Does the response contain the correct region?
2. Does the response contain the correct sales value?

The response will be accepted only when both values match.

### Insight

The output guardrail prevents an incorrect or invented result from reaching the user.

In [12]:
def output_guardrail(response_text, verified_result):

    verified_region = verified_result['region']
    verified_sales = str(verified_result['sales'])

    response_text = response_text.replace(',', '')

    if verified_region in response_text and verified_sales in response_text:
        return True

    return False

### Test the Output Guardrail

We will test three situations:

1. Correct region and correct sales
2. Wrong region
3. Wrong sales

The guardrail should return:

- `True` for the correct response
- `False` for incorrect responses

In [13]:
print(
    output_guardrail(
        "The South region recorded 20,000 in sales.",
        result
    )
)

print(
    output_guardrail(
        "The North region recorded 20,000 in sales.",
        result
    )
)

print(
    output_guardrail(
        "The South region recorded 15,000 in sales.",
        result
    )
)

True
False
False


### Interpretation

The output guardrail correctly accepts the response containing the verified region and sales value.

It rejects responses containing an incorrect region or incorrect sales value.

### Insight

The LLM response is not automatically trusted. It must match the verified Python result before it can be shown to the user.

## Step 8 — Final Integrated AI Data Analyst Agent

The individual components are now combined into one workflow.

### Workflow

User Question
↓
Input Guardrail
↓
Attempt Limit
↓
Python/Pandas Calculation
↓
Verified Structured Result
↓
Gemini Explanation
↓
Output Guardrail
↓
Verified Response

### Key Principle

Python is responsible for calculating the result.

Gemini is responsible for explaining the result.

The input and output guardrails control the agent's workflow.

In [16]:
attempts = 0
max_attempts = 3

while attempts < max_attempts:

    question = input("Ask a data analysis question (type exit to exit): ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    # Input guardrail
    if not input_guardrail(question):

        attempts += 1
        remaining = max_attempts - attempts

        if remaining > 0:
            print("Request rejected.")
            print(f"Remaining attempts: {remaining}")
            continue

        else:
            print("Maximum attempts reached. Program stopped.")
            break

    print("Request accepted.")

    # Python calculation
    highest_sales = df.loc[df["Sales"].idxmax()]

    # Structured verified result
    result = {
        "region": highest_sales["Region"],
        "sales": highest_sales["Sales"]
    }

    # Prompt Gemini
    prompt = f"""
    You are a data analyst.

    The Python calculation produced this verified result:
    {result}

    Explain the finding in one simple sentence.
    Do not change or invent the region or sales value.
    """

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )

    # Output guardrail
    if output_guardrail(response.text, result):

        print("Verified response:")
        print(response.text)
        break

    else:

        attempts += 1
        remaining = max_attempts - attempts

        print("Response rejected by output guardrail.")

        if remaining > 0:
            print(f"Remaining attempts: {remaining}")
        else:
            print("Maximum attempts reached. Program stopped.")
            break

Ask a data analysis question (type exit to exit): exit
Goodbye!


### Interpretation

The final agent successfully combines the individual components into one controlled workflow.

A valid question passes through the input guardrail, Python calculates the verified result, Gemini explains the result, and the output guardrail checks the response before it is displayed.

Invalid questions are rejected and limited by the attempt counter.

### Insight

The agent is not simply sending a user question directly to an LLM.

It follows a controlled process:

**Validate → Calculate → Explain → Verify → Respond**

## Step 9 — Failure Log

During development, several failures were identified and fixed.

### Failure 1 — API Quota Error

**Observed behavior:**  
The Gemini API returned a `429 RESOURCE_EXHAUSTED` error.

**Cause:**  
The free-tier request quota had been exceeded.

**Lesson:**  
API failures should be handled with a short, user-friendly message instead of exposing a long technical error.

---

### Failure 2 — Input Guardrail Loop

**Observed behavior:**  
After an invalid question, the program continued asking for another question.

**Cause:**  
The loop required explicit control using `continue` and an attempt counter.

**Fix:**  
Added a maximum of three attempts and used `break` when the limit was reached.

**Lesson:**  
Loop control is important for preventing an agent from running indefinitely.

---

### Failure 3 — Output Guardrail Rejected a Correct Response

**Observed behavior:**  
Gemini returned a correct sales value such as `20,000`, but the guardrail expected `20000`.

**Cause:**  
The comparison depended on the exact text format.

**Fix:**  
Removed commas from the response before checking the sales value.

**Lesson:**  
Validation should account for harmless differences in formatting.

---

### Failure 4 — Agent Continued After a Successful Response

**Observed behavior:**  
The agent displayed the verified answer and then asked for another question.

**Cause:**  
The `while` loop started another iteration after successfully displaying the response.

**Fix:**  
Added `break` after the verified response.

**Lesson:**  
`break` stops the workflow when the current task is successfully completed, while `continue` is used when another attempt is required.

# Lesson 5 — Final Insight

In this lesson, I built a guarded AI Data Analyst Agent.

The agent follows a controlled workflow:

**Validate → Calculate → Explain → Verify → Respond**

Python/Pandas performs the numerical calculation and produces the verified result.

Gemini converts the verified result into a simple natural-language explanation.

The input guardrail prevents unrelated questions from entering the workflow.

The output guardrail checks whether the generated response matches the verified result.

The development process also demonstrated why testing is important. API quota errors, loop behavior, number formatting, and output validation all required testing and correction.

### Key Learning

An AI agent should not blindly trust the LLM.

The LLM should work within a controlled workflow where important results are calculated and verified using reliable tools.